<a href="https://colab.research.google.com/github/krasnovaov81-coder/netology/blob/main/itog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd

# --- НАСТРОЙКИ (config) ---
DATA_DIR = "data"
REPORTS_DIR = "reports"
LOGS_DIR = "logs"
OUTPUT_REPORT_FILENAME = "summary_report.csv"
LOG_FILENAME = "errors.log"

STATUS_COLUMN = "status"
DELIVERED_STATUS_VALUE = "Delivered"
# --- КОНЕЦ НАСТРОЕК ---


class OrderAnalyzer:
    """
    Класс для анализа данных о заказах из CSV-файлов.
    """
    def __init__(self):
        self.results = []
        self.error_count = 0
        self.success_count = 0

        # Создаем директории, если их нет
        os.makedirs(REPORTS_DIR, exist_ok=True)
        os.makedirs(LOGS_DIR, exist_ok=True)
        self.log_path = os.path.join(LOGS_DIR, LOG_FILENAME)

    def _log_error(self, filename: str, error_msg: str):
        """Записывает информацию об ошибке в лог-файл."""
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(f"Файл: {filename} | Ошибка: {error_msg}\n")

    def _process_file(self, filepath: str):
        """Обрабатывает один CSV-файл."""
        filename = os.path.basename(filepath)
        try:
            df = pd.read_csv(filepath)

            # Проверка наличия обязательных колонок
            required_cols = {STATUS_COLUMN, "total_amount"}
            if not required_cols.issubset(df.columns):
                raise ValueError(f"Отсутствуют обязательные колонки: {required_cols}")

            # Фильтрация доставленных заказов
            delivered_df = df[df[STATUS_COLUMN] == DELIVERED_STATUS_VALUE]
            if delivered_df.empty:
                # Если нет доставленных заказов, просто пропускаем файл без ошибки
                return

            # Расчет метрик
            total_revenue = delivered_df["total_amount"].sum()
            average_check = delivered_df["total_amount"].mean()
            orders_count = len(delivered_df)

            self.results.append({
                "filename": filename,
                "total_revenue": round(total_revenue, 2),
                "average_check": round(average_check, 2),
                "orders_count": orders_count,
            })
            self.success_count += 1

        except Exception as e:
            # Логируем любую ошибку и продолжаем работу со следующим файлом
            self._log_error(filename, str(e))
            self.error_count += 1

    def process_all_files(self):
        """Находит и обрабатывает все CSV-файлы в папке DATA_DIR."""
        if not os.path.exists(DATA_DIR):
            print(f"Папка '{DATA_DIR}' не найдена. Создайте её и поместите туда CSV-файлы.")
            return

        for file in os.listdir(DATA_DIR):
            if file.lower().endswith(".csv"):
                filepath = os.path.join(DATA_DIR, file)
                self._process_file(filepath)

    def save_report(self):
        """Сохраняет итоговый отчет в CSV-файл."""
        if not self.results:
            print("Нет данных для формирования отчета (возможно, нет файлов со статусом 'Delivered').")
            return

        report_path = os.path.join(REPORTS_DIR, OUTPUT_REPORT_FILENAME)
        df_report = pd.DataFrame(self.results)
        df_report.to_csv(report_path, index=False)

    def print_summary(self):
        """Выводит итоговую статистику в консоль."""
        print(f"Обработано файлов: {self.success_count}")
        print(f"Файлов с ошибками: {self.error_count}")


def main():
    """Точка входа в приложение."""
    analyzer = OrderAnalyzer()
    analyzer.process_all_files()
    analyzer.save_report()
    analyzer.print_summary()


if __name__ == "__main__":
    main()

Обработано файлов: 2
Файлов с ошибками: 0
